## DATA LOADING

In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="GeVk2gPJo9OzY6HJRsaz")
project = rf.workspace("streetlight-detection").project("sodioum-only-jkq3f")
version = project.version(7)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 126.2 MB/s eta 0:00:0000:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquer


Extracting Dataset Version Zip to Sodioum-Only-7 in yolov8:: 100%|██████████| 20012/20012 [00:16<00:00, 1208.67it/s] 


In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key="GeVk2gPJo9OzY6HJRsaz")
project = rf.workspace("godspeed-yqpeo").project("damaged-lights")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Damaged-Lights-1 in yolov8:: 100%|██████████| 3355/3355 [00:00<00:00, 7628.69it/s]


In [6]:
import yaml
import glob

# Find all data.yaml files downloaded from Roboflow in the Colab environment
yaml_files = glob.glob('/content/**/data.yaml', recursive=True)

print("🔍 Inspecting Dataset Class Mappings...\n")

for file in yaml_files:
    try:
        with open(file, 'r') as f:
            data = yaml.safe_load(f)
            # Extract just the folder name for readability
            dataset_name = file.split('/content/')[1].split('/')[0]
            print(f"--- Dataset: {dataset_name} ---")
            print(f"Classes: {data.get('names', 'No names found')}\n")
    except Exception as e:
        print(f"Could not read {file}: {e}")

🔍 Inspecting Dataset Class Mappings...



## MASTER DATASET

In [7]:
import os
import shutil
import yaml

# Define paths (Adjust these root names if your Colab folders are named slightly differently)
godspeed_root = '/kaggle/working/Damaged-Lights-1'
sodium_root = '/kaggle/working/Sodioum-Only-7'
unified_root = '/kaggle/working/streetlights_unified'
# 1. Define our new Master Class Indices
# 0: light_on, 1: light_off, 2: damaged_fixture

# Mapping dictionaries (Old Class Index -> New Class Index)
godspeed_map = {
    '0': '2', # 'Not Working' -> damaged_fixture
    '1': '0'  # 'Working' -> light_on
}

sodium_map = {
    '0': '1', # 'sodium_off' -> light_off
    '1': '0'  # 'sodium_on' -> light_on
}

splits = ['train', 'valid', 'test']

def process_dataset(source_root, class_mapping):
    if not os.path.exists(source_root):
        print(f"Directory not found: {source_root}")
        return

    for split in splits:
        # Handling Roboflow's standard export structure
        src_images = os.path.join(source_root, split, 'images')
        src_labels = os.path.join(source_root, split, 'labels')

        dest_images = os.path.join(unified_root, split, 'images')
        dest_labels = os.path.join(unified_root, split, 'labels')

        os.makedirs(dest_images, exist_ok=True)
        os.makedirs(dest_labels, exist_ok=True)

        if not os.path.exists(src_labels):
            continue

        for label_file in os.listdir(src_labels):
            if not label_file.endswith('.txt'): continue

            # Read old labels, map classes, write to new unified folder
            with open(os.path.join(src_labels, label_file), 'r') as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts: continue
                old_class = parts[0]
                if old_class in class_mapping:
                    new_class = class_mapping[old_class]
                    new_lines.append(f"{new_class} " + " ".join(parts[1:]) + "\n")

            # Save new label file
            with open(os.path.join(dest_labels, label_file), 'w') as f:
                f.writelines(new_lines)

            # Copy corresponding image
            img_name = label_file.replace('.txt', '.jpg') # Default Roboflow output
            if not os.path.exists(os.path.join(src_images, img_name)):
                # Catch case where image is .png
                img_name = label_file.replace('.txt', '.png')

            try:
                shutil.copy(os.path.join(src_images, img_name), os.path.join(dest_images, img_name))
            except FileNotFoundError:
                pass # Skip if image is missing for some reason

print("🔄 Starting Dataset Unification...")
process_dataset(godspeed_root, godspeed_map)
process_dataset(sodium_root, sodium_map)

# 2. Generate Master data.yaml for YOLOv8
master_yaml = {
    'path': unified_root,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'light_on',
        1: 'light_off',
        2: 'damaged_fixture'
    }
}

with open(os.path.join(unified_root, 'master_data.yaml'), 'w') as f:
    yaml.dump(master_yaml, f, default_flow_style=False)

print(f"✅ Unification Complete! Master dataset created at: {unified_root}")
print(f"✅ master_data.yaml generated. Ready for PyTorch.")

🔄 Starting Dataset Unification...
✅ Unification Complete! Master dataset created at: /kaggle/working/streetlights_unified
✅ master_data.yaml generated. Ready for PyTorch.


## TRAINING

In [8]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 49.4 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.7 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 9.5 MB/s eta 0:00:000:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but 

In [9]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [10]:
import os
import shutil

train_images = '/kaggle/working/streetlights_unified/train/images'
train_labels = '/kaggle/working/streetlights_unified/train/labels'
cache_path = '/kaggle/working/streetlights_unified/train/labels.cache'
# 1. Identify all files containing the minority class (2: damaged_fixture)
damaged_files = []
for label_file in os.listdir(train_labels):
    if label_file.endswith('.txt'):
        with open(os.path.join(train_labels, label_file), 'r') as f:
            # Check if any line in the text file starts with class '2'
            if any(line.startswith('2 ') for line in f):
                damaged_files.append(label_file)

print(f"Found {len(damaged_files)} damaged images. Duplicating 7x to balance dataset...")

# 2. Oversample the minority class physically
for i in range(7):
    for label_file in damaged_files:
        img_file = label_file.replace('.txt', '.jpg')

        new_label = label_file.replace('.txt', f'_copy{i}.txt')
        new_img = img_file.replace('.jpg', f'_copy{i}.jpg')

        # Copy the annotation label
        shutil.copy(os.path.join(train_labels, label_file), os.path.join(train_labels, new_label))

        # Copy the image (Fallback to .png if .jpg doesn't exist)
        src_img = os.path.join(train_images, img_file)
        if not os.path.exists(src_img):
             src_img = src_img.replace('.jpg', '.png')
             new_img = new_img.replace('.jpg', '.png')

        if os.path.exists(src_img):
            shutil.copy(src_img, os.path.join(train_images, new_img))

# 3. Purge the old cache so YOLO is forced to rescan the new balanced folder
cache_path = '/content/streetlights_unified/train/labels.cache'
if os.path.exists(cache_path):
    os.remove(cache_path)

print("✅ Physical oversampling complete! The dataset is mathematically balanced.")

Found 843 damaged images. Duplicating 7x to balance dataset...
✅ Physical oversampling complete! The dataset is mathematically balanced.


In [11]:
import os
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

yaml_path = '/kaggle/working/streetlights_unified/master_data.yaml'

print("🚀 Launching balanced, error-free training loop...")

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=320,
    batch=128,
    workers=0,
    cache=True,
    amp=True,

    # Spatial augmentations re-enabled to prevent the model from memorizing the duplicated images
    mosaic=1.0,
    mixup=0.15,

    patience=8,
    device=0,
    project='CityLens_Hackathon',
    name='streetlight_balanced_run'
)

print("\n✅ Balanced training complete!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Launching balanced, error-free training loop...
Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/streetlights_unified/master_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, frac

In [3]:
# 1. Relocate your prize-winning weights to the safe root directory
!cp /kaggle/working/runs/detect/CityLens_Hackathon/streetlight_balanced_run/weights/best.pt /kaggle/working/best_model.pt

# 2. Nuke the massive training and validation folders (Training is done!)
!rm -rf /kaggle/working/streetlights_unified/train
!rm -rf /kaggle/working/streetlights_unified/valid

# 3. Nuke the training logs (clears gigabytes of heavy last.pt files and cache)
!rm -rf /kaggle/working/runs

# 4. Delete the half-finished deliverables folder so we can start fresh
!rm -rf /kaggle/working/Hackathon_Static_Deliverables

# 5. Verify the restored disk space
!df -h /kaggle/working/

cp: error writing '/kaggle/working/best_model.pt': No space left on device
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   12G  7.8G  61% /kaggle/working


## OUTPUT RESULTS

In [6]:
import os
from ultralytics import YOLO

# Point directly to the file you just manually uploaded
model = YOLO('/kaggle/input/datasets/buriburiburi/weights/best.pt')

# Point to your test images (which we saved during the purge)
test_source = '/kaggle/working/streetlights_unified/test/images' 

print("📸 Regenerating deliverables for exactly 500 images...")

results = model.predict(
    source=test_source,
    conf=0.45,             
    imgsz=320,             
    batch=4,               
    half=False,            
    stream=True,           
    save=True,             
    save_txt=True,         
    project='/kaggle/working/',
    name='Hackathon_Static_Deliverables'
)

# Loop through the generator and break exactly at 500
count = 0
for r in results:
    count += 1
    if count >= 500:
        break

print(f"\n✅ Deliverables generated for {count} images! You can now zip and download the folder.")

📸 Regenerating deliverables for exactly 500 images...

image 1/1000 /kaggle/working/streetlights_unified/test/images/20250929_151822_jpg.rf.0eeb5db2efffaaeff9f32f48a8f3de97.jpg: 320x192 1 light_off, 3.5ms
image 2/1000 /kaggle/working/streetlights_unified/test/images/20250929_151823_jpg.rf.8b16fc55131dd6bc63017963a1ed8eb8.jpg: 320x192 1 light_off, 3.5ms
image 3/1000 /kaggle/working/streetlights_unified/test/images/20250929_151824_jpg.rf.a0b832f8cf083b0bdfb8d014e3ec64c7.jpg: 320x192 1 light_off, 3.5ms
image 4/1000 /kaggle/working/streetlights_unified/test/images/20250929_151825_jpg.rf.c06f5b5cb6aff34ae271b2fe6b23e812.jpg: 320x192 1 light_off, 3.5ms
image 5/1000 /kaggle/working/streetlights_unified/test/images/20250929_151826_jpg.rf.47bc7173479c9a2a6beeb60120d73008.jpg: 320x192 1 light_off, 3.3ms
image 6/1000 /kaggle/working/streetlights_unified/test/images/20250929_151827_jpg.rf.b052faec2cea488d724eb7795139e1ca.jpg: 320x192 1 light_off, 3.3ms
image 7/1000 /kaggle/working/streetlights_uni

In [7]:
import shutil
from IPython.display import FileLink

# Zip the newly generated deliverables folder
shutil.make_archive('Hackathon_Static_Submission', 'zip', '/kaggle/working/Hackathon_Static_Deliverables')

# Click the link below to download your final ZIP file
FileLink(r'Hackathon_Static_Submission.zip')

/kaggle/working/Hackathon_Static_Submission.zip

## FLICKER DETECTION MODEL

In [ ]:
# Clone the repository directly into Colab's fast /content/ storage
!git clone https://github.com/Team16Project/Street-Light-Dataset.git

# Verify the download and inspect the folder structure to locate your .mp4 files
!ls -la Street-Light-Dataset

In [2]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.2 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you 

In [7]:
import cv2
import numpy as np
import os
import glob
import pandas as pd
from collections import deque
from ultralytics import YOLO

# --- 1. KAGGLE PATH SETUP ---
INPUT_DIR = '/kaggle/working/Street-Light-Dataset/Raw Dataset/Flicker'
OUTPUT_FRAME_DIR = '/kaggle/working/Deliverable_Frames'
os.makedirs(OUTPUT_FRAME_DIR, exist_ok=True)

# Load your hardware-optimized static model
model = YOLO('/kaggle/input/datasets/buriburiburi/weights/best.pt')

# --- 2. TEMPORAL PARAMETERS ---
WINDOW_SIZE = 15         # Sliding window of 15 frames
VARIANCE_THRESHOLD = 300 # Adjust if necessary based on exposure

def get_light_id(x1, y1):
    """Spatial hashing to track the same streetlight across frames"""
    return f"{int(x1//50)}_{int(y1//50)}"

# --- 3. BATCH PROCESSING LOOP ---
video_files = glob.glob(os.path.join(INPUT_DIR, '*.mp4'))
analytics_log = []

print(f"Found {len(video_files)} videos. Starting batch inference...")

for video_path in video_files:
    filename = os.path.basename(video_path)
    cap = cv2.VideoCapture(video_path)
    
    light_histories = {}
    flicker_detected = False
    frame_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        results = model(frame, stream=True, verbose=False) # Suppress YOLO printouts for cleaner logs
        
        for r in results:
            for box in r.boxes:
                # Extract coordinates
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                # Crop to Region of Interest
                roi = frame[y1:y2, x1:x2]
                if roi.size == 0:
                    continue

                # Calculate grayscale intensity
                gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
                mean_intensity = np.mean(gray_roi)

                l_id = get_light_id(x1, y1)
                if l_id not in light_histories:
                    light_histories[l_id] = deque(maxlen=WINDOW_SIZE)
                
                light_histories[l_id].append(mean_intensity)

                # --- 4. FLICKER CLASSIFICATION ---
                if len(light_histories[l_id]) == WINDOW_SIZE:
                    variance = np.var(light_histories[l_id])
                    
                    if variance > VARIANCE_THRESHOLD and not flicker_detected:
                        flicker_detected = True
                        
                        # Draw bounding box and label for the deliverable
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
                        cv2.putText(frame, f"FLICKERING (Var: {variance:.1f})", (x1, y1 - 10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                        
                        # Save the specific frame to disk (Avoids saving massive videos)
                        output_frame_path = os.path.join(OUTPUT_FRAME_DIR, f"detected_{filename}.jpg")
                        cv2.imwrite(output_frame_path, frame)
                        print(f"Flicker caught in {filename} at frame {frame_count}. Frame saved.")

        # Optional: Stop processing this video early if flicker is already found 
        # to save Kaggle GPU compute time. Remove the next two lines if you want to 
        # process the whole video regardless.
        if flicker_detected:
            break

    cap.release()
    
    # Log the result for the Bonus Analytics CSV
    analytics_log.append({
        "Video_File": filename,
        "Flicker_Status": "Detected" if flicker_detected else "Stable",
        "Frames_Processed": frame_count
    })

# --- 5. GENERATE ANALYTICS DELIVERABLE ---
df = pd.DataFrame(analytics_log)
csv_path = '/kaggle/working/flicker_analytics.csv'
df.to_csv(csv_path, index=False)

print("\n--- BATCH PROCESSING COMPLETE ---")
print(f"Annotated frames saved to: {OUTPUT_FRAME_DIR}")
print(f"Analytics report saved to: {csv_path}")

Found 22 videos. Starting batch inference...
Flicker caught in 20240309_190900.mp4 at frame 103. Frame saved.
Flicker caught in VID20240307184842.mp4 at frame 139. Frame saved.
Flicker caught in 20240301_192405.mp4 at frame 45. Frame saved.
Flicker caught in VID-20240303-WA0003.mp4 at frame 100. Frame saved.
Flicker caught in VID-20240303-WA0001.mp4 at frame 163. Frame saved.
Flicker caught in VID-20240303-WA0005.mp4 at frame 57. Frame saved.
Flicker caught in VID-20240303-WA0004.mp4 at frame 19. Frame saved.

--- BATCH PROCESSING COMPLETE ---
Annotated frames saved to: /kaggle/working/Deliverable_Frames
Analytics report saved to: /kaggle/working/flicker_analytics.csv
